# tidypr: fine-tune on Kaggle (free T4) with Unsloth

**One-time setup**
1. Kaggle account must be **phone-verified** (needed for GPU + Internet).
2. On your Mac, zip just the dataset: `cd tidypr/data && zip -r tidypr-data.zip out`
3. kaggle.com → **Datasets → New Dataset** → upload `tidypr-data.zip`, title `tidypr-data`, keep it **Private**.
4. kaggle.com → **Create → New Notebook** → **File → Import Notebook** → upload this file. In the right panel:
   - **Add Input** → your `tidypr-data` dataset
   - Settings → Accelerator → **GPU T4 x1** (not P100: Unsloth needs a newer GPU)
   - Settings → Internet → **On**
   - Add-ons → Secrets → add `HF_TOKEN` (Hugging Face *write* token), toggle it on for this notebook
5. Edit the config cell, then **Run All**.

The code is cloned from GitHub; the dataset comes from your Kaggle input. (Uploading the whole project zip instead also works: set `GIT_REPO = ""`.)

In [ ]:
# ---- config ------------------------------------------------------------------
HF_REPO   = "bunnyweb/tidypr-1.5b-gguf"   # <your HF username>/<model name>; "" = don't push
BASE      = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
MAX_STEPS = 30        # 30 = ~5 min smoke test. Set to -1 for the full run (2 epochs).
EPOCHS    = 1         # v2 data: 1 epoch; more just memorises the templates
GIT_REPO  = "https://github.com/thebunnyweb/tidypr"   # "" = use an uploaded project zip
DATA_HF   = "bunnyweb/tidypr-data"   # master dataset on HF (private); "" = use the Kaggle input dataset

assert "your-hf-username" not in HF_REPO, "Set HF_REPO to your real Hugging Face username first"

In [ ]:
import os, shutil, glob, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU
WORK = "/kaggle/working/tidypr"

if GIT_REPO:
    if not os.path.exists(WORK):
        subprocess.run(["git", "clone", "-q", GIT_REPO, WORK], check=True)
else:
    hits = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    assert hits, "No GIT_REPO and no uploaded project found under /kaggle/input"
    shutil.copytree(os.path.dirname(hits[0]), WORK, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns(".venv", "runs"))
os.chdir(WORK)

# dataset: the folder under /kaggle/input that holds train.jsonl + valid.jsonl
if DATA_HF and not os.path.exists("data/out/train.jsonl"):
    from huggingface_hub import snapshot_download
    from kaggle_secrets import UserSecretsClient
    d = snapshot_download(DATA_HF, repo_type="dataset", allow_patterns=["master/*"],
                          token=UserSecretsClient().get_secret("HF_TOKEN"))
    shutil.copytree(os.path.join(d, "master"), "data/out", dirs_exist_ok=True)
    print("dataset from HF:", DATA_HF)
if not os.path.exists("data/out/train.jsonl"):
    hits = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
    assert hits, "Add your tidypr-data dataset as an input (it must contain train.jsonl and valid.jsonl)"
    shutil.copytree(os.path.dirname(hits[0]), "data/out", dirs_exist_ok=True)

for f in ("data/out/train.jsonl", "data/out/valid.jsonl"):
    print(f, sum(1 for _ in open(f)), "examples")
print(open("data/out/stats.json").read() if os.path.exists("data/out/stats.json") else "")
!git log --oneline -1

In [ ]:
!pip install -q unsloth
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("No HF_TOKEN secret — model will be saved locally only.", e)
    HF_REPO = ""

In [ ]:
push = f"--push {HF_REPO}" if HF_REPO and os.environ.get("HF_TOKEN") else ""
!python training/train_unsloth.py --data data/out --out runs/unsloth --base {BASE} \
    --epochs {EPOCHS} --max-steps {MAX_STEPS} --gguf q8_0 {push}

**Check it works** on the sample PR and the validation set (CPU llama.cpp, same as the GitHub Action will use).

In [ ]:
!pip install -q "llama-cpp-python>=0.3.0" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
GGUF = sorted(glob.glob("runs/unsloth/**/*Q8_0*.gguf", recursive=True))[0]   # not the big F16 one
print(GGUF)
!tidypr review --diff data/seed/sample_pr.diff --no-rules --backend llamacpp --model {GGUF}

In [ ]:
print("=== rules baseline ===")
!python eval/evaluate.py --data data/out/valid.jsonl --limit 100
print("=== fine-tuned model alone ===")
!python eval/evaluate.py --data data/out/valid.jsonl --limit 100 --no-rules --backend llamacpp --model {GGUF}
if os.path.exists("data/gold/gold.jsonl"):
    print("=== GOLD SET (real hunks you labelled): rules ===")
    !python eval/evaluate.py --data data/gold/gold.jsonl
    print("=== GOLD SET: fine-tuned model alone (the number that matters) ===")
    !python eval/evaluate.py --data data/gold/gold.jsonl --no-rules --backend llamacpp --model {GGUF}
if os.path.exists("data/out/test.jsonl"):
    print("=== REAL TEST SPLIT (held-out repos): fine-tuned model alone ===")
    !python eval/evaluate.py --data data/out/test.jsonl --no-rules --backend llamacpp --model {GGUF}

**Download** (if you didn't push to Hugging Face): the zip appears under *Output* on the right.

In [ ]:
!cd runs/unsloth && zip -qr /kaggle/working/tidypr-model.zip lora && zip -qj /kaggle/working/tidypr-model.zip {GGUF.replace("runs/unsloth/","")} && ls -lh /kaggle/working/*.zip